# TFT Scouting — Notebook 1: Top Players

**Objetivo:** Obtener la lista de jugadores Challenger + Grandmaster + Master de EUW,
resolver sus PUUIDs y guardar el resultado como base para el resto de notebooks.

**Routing:**
- League / Summoner endpoints → `euw1.api.riotgames.com` (platform routing)
- Match endpoints → `europe.api.riotgames.com` (regional routing)

**Rate limits (dev key):** 20 req/s — 100 req/2min

In [1]:
# ── Dependencias ─────────────────────────────────────────────────────────────
# pip install requests pandas tqdm

import os
import time
import requests
import pandas as pd
from tqdm.notebook import tqdm

API_KEY = 'RGAPI-c05d22b7-d146-4011-b5af-6307a0397f1a'

PLATFORM = 'euw1.api.riotgames.com'
REGION   = 'europe.api.riotgames.com'

HEADERS = {'X-Riot-Token': API_KEY}

In [2]:
# ── Helper con rate-limit simple ──────────────────────────────────────────────

def get(url, params=None, retries=3):
    """GET con retry en 429 y pausa automática."""
    for attempt in range(retries):
        r = requests.get(url, headers=HEADERS, params=params)
        if r.status_code == 200:
            return r.json()
        elif r.status_code == 429:
            wait = int(r.headers.get('Retry-After', 5))
            print(f'Rate limited — esperando {wait}s')
            time.sleep(wait)
        elif r.status_code == 404:
            return None
        else:
            print(f'Error {r.status_code}: {r.text}')
            time.sleep(1)
    return None

In [3]:
# ── 1. Obtener Challenger ─────────────────────────────────────────────────────
# Devuelve ~200 jugadores con summonerId y LP

url = f'https://{PLATFORM}/tft/league/v1/challenger'
challenger_data = get(url, params={'queue': 'RANKED_TFT'})

print(f"Challenger players: {len(challenger_data['entries'])}")
print(challenger_data['entries'][0])  # Ver estructura

Challenger players: 200
{'puuid': 'ku6-OAKXILJMi9QAV-oQgTP7I2ZUQzWETi9LB2qLI-QYA7rbm9wkpl3zQsGL5-aEu513_Tfunrqzhg', 'leaguePoints': 1904, 'rank': 'I', 'wins': 240, 'losses': 115, 'veteran': True, 'inactive': False, 'freshBlood': False, 'hotStreak': True}


In [4]:
# ── 2. Obtener Grandmaster ────────────────────────────────────────────────────

url = f'https://{PLATFORM}/tft/league/v1/grandmaster'
gm_data = get(url, params={'queue': 'RANKED_TFT'})

print(f"Grandmaster players: {len(gm_data['entries'])}")
print(gm_data['entries'][0])

Grandmaster players: 400
{'puuid': 'F0K-k4tWfz6p7gMBqon96qbDV1YYjgbv34rCFrNjlpJxdKxJ3Hg8xk5lkBSGOu3cxj2xYGNjGRtxBA', 'leaguePoints': 1212, 'rank': 'I', 'wins': 441, 'losses': 381, 'veteran': False, 'inactive': False, 'freshBlood': True, 'hotStreak': False}


In [5]:
# ── 3. Combinar y ordenar por LP ──────────────────────────────────────────────

all_entries = challenger_data['entries'] + gm_data['entries']

df_players = pd.DataFrame(all_entries)[['puuid', 'leaguePoints', 'wins', 'losses']]
df_players['tier'] = ['CHALLENGER'] * len(challenger_data['entries']) + \
                     ['GRANDMASTER'] * len(gm_data['entries'])
df_players = df_players.sort_values('leaguePoints', ascending=False).reset_index(drop=True)

print(f'Total jugadores: {len(df_players)}')
df_players.head(10)

Total jugadores: 600


,puuid,leaguePoints,wins,losses,tier
0,ku6-OAKXILJMi9QAV-oQgTP7I2ZUQzWETi9LB2qLI-QYA7...,1904,240,115,CHALLENGER
1,WwCqAGLhx8AmD_O-2pW_ZaKzPuvsXINaXQ_gugz9VlljGe...,1706,722,548,CHALLENGER
2,gp22GSIE5qqK95O1ZcnkoO4gHku-nT3JDIZqJaMCOianXa...,1681,339,249,CHALLENGER
3,E_YnCFtaEQqxmkAe-I10Op9W3AkpwxywsArmbm-gs4sEl8...,1531,253,166,CHALLENGER
4,qIZwsHPHsbwgiunHe5JA6T5SKoje75DKfpRW3mvVEcIiO6...,1531,292,157,CHALLENGER
5,xfNbaAvQ2W5_8vQPoonoo_H5NrUl0PO6XndHQEx3wgro4M...,1521,137,49,CHALLENGER
6,3gxNs9Q7ycFfNgQLn-1WwO-oM867LzSNSVpZbaHBMtP-S0...,1513,242,147,CHALLENGER
7,ErleM9HM35cifY76jutT1rg4qHstuw5d5laWTTwfSDNBAz...,1510,187,111,CHALLENGER
8,HKYZTjSVMFLwZiWFsh1TtLwMatHp5NQnhsQgNvcyCu97x2...,1510,272,188,CHALLENGER
9,KdYLzfmqJoCw3P-arYYKGLQFzLy6zjEVt08OZ5iPiiie61...,1510,234,167,CHALLENGER


In [6]:
# ── 4. Para la POC, limitamos a top 200 por LP ───────────────────────────────

N_PLAYERS = 200
df_poc = df_players.head(N_PLAYERS).copy()
print(f'LP mínimo en muestra: {df_poc["leaguePoints"].min()}')
df_poc.head()

LP mínimo en muestra: 1084


,puuid,leaguePoints,wins,losses,tier
0,ku6-OAKXILJMi9QAV-oQgTP7I2ZUQzWETi9LB2qLI-QYA7...,1904,240,115,CHALLENGER
1,WwCqAGLhx8AmD_O-2pW_ZaKzPuvsXINaXQ_gugz9VlljGe...,1706,722,548,CHALLENGER
2,gp22GSIE5qqK95O1ZcnkoO4gHku-nT3JDIZqJaMCOianXa...,1681,339,249,CHALLENGER
3,E_YnCFtaEQqxmkAe-I10Op9W3AkpwxywsArmbm-gs4sEl8...,1531,253,166,CHALLENGER
4,qIZwsHPHsbwgiunHe5JA6T5SKoje75DKfpRW3mvVEcIiO6...,1531,292,157,CHALLENGER


In [7]:
# ── 5. SKIP — PUUID ya viene en la respuesta, no hay que resolver nada ────────
print(f'PUUIDs disponibles: {df_poc["puuid"].notna().sum()} / {len(df_poc)}')

PUUIDs disponibles: 200 / 200


In [8]:
# ── 6. Guardar para notebook 2 ───────────────────────────────────────────────

df_poc.to_csv('top_players.csv', index=False)
print('Guardado top_players.csv')
df_poc.head()

Guardado top_players.csv


,puuid,leaguePoints,wins,losses,tier
0,ku6-OAKXILJMi9QAV-oQgTP7I2ZUQzWETi9LB2qLI-QYA7...,1904,240,115,CHALLENGER
1,WwCqAGLhx8AmD_O-2pW_ZaKzPuvsXINaXQ_gugz9VlljGe...,1706,722,548,CHALLENGER
2,gp22GSIE5qqK95O1ZcnkoO4gHku-nT3JDIZqJaMCOianXa...,1681,339,249,CHALLENGER
3,E_YnCFtaEQqxmkAe-I10Op9W3AkpwxywsArmbm-gs4sEl8...,1531,253,166,CHALLENGER
4,qIZwsHPHsbwgiunHe5JA6T5SKoje75DKfpRW3mvVEcIiO6...,1531,292,157,CHALLENGER
